In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interactive, IntSlider, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, GridBox
from IPython.display import display

# ============================================================
# CSS FOR HORIZONTAL RADIO BUTTONS
# ============================================================

display(HTML("""
<style>
.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    align-items: center !important;
    gap: 18px !important;
}
.horizontal-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}
.horizontal-radio > label {
    display: none !important;
}
</style>
"""))

# ============================================================
# RANDOM DATA
# ============================================================

np.random.seed(117)

M_max = 100
N = 600
T_end = 10.0

t = np.linspace(0.0, T_end, N)

noise_bank = np.random.randn(M_max, N)

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_stochastic_differential_equation(mean_type='Constant', mean_level=1.0, tau=1.0, noise_std=0.8, M=40):

    # --------------------------------------------------------
    # INPUT MEAN
    # --------------------------------------------------------

    if mean_type == 'Constant':

        mu_x = mean_level * np.ones_like(t)

        mean_name = 'Constant Input Mean'

    else:

        f0 = 0.25

        mu_x = mean_level * np.sin(2.0 * np.pi * f0 * t)

        mean_name = 'Sinusoidal Input Mean'

    # --------------------------------------------------------
    # STOCHASTIC INPUT REALIZATIONS
    #
    # x_i(t) = mu_x(t) + noise
    # --------------------------------------------------------

    X = mu_x[None, :] + noise_std * noise_bank[:M, :]

    # --------------------------------------------------------
    # FIRST-ORDER LTI SYSTEM
    #
    # tau dy/dt + y = x(t)
    #
    # H(s) = 1 / (tau s + 1)
    # --------------------------------------------------------

    system = signal.TransferFunction([1.0], [tau, 1.0])

    # --------------------------------------------------------
    # OUTPUT REALIZATIONS
    # --------------------------------------------------------

    Y = np.zeros_like(X)

    for m in range(M):

        _, Y[m, :], _ = signal.lsim(system, U=X[m, :], T=t)

    # --------------------------------------------------------
    # ESTIMATED ENSEMBLE MEANS
    # --------------------------------------------------------

    estimated_mu_x = np.mean(X, axis=0)

    estimated_mu_y = np.mean(Y, axis=0)

    # --------------------------------------------------------
    # THEORETICAL OUTPUT MEAN
    #
    # tau d(mu_y)/dt + mu_y = mu_x(t)
    # --------------------------------------------------------

    _, theoretical_mu_y, _ = signal.lsim(system, U=mu_x, T=t)

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 4.6))

    number_to_show = min(10, M)

    # ========================================================
    # GRAPH 1:
    # STOCHASTIC INPUT
    # ========================================================

    for m in range(number_to_show):

        ax1.plot(t, X[m, :], linewidth=0.65, alpha=0.22)

    ax1.plot(t, estimated_mu_x, linewidth=2.0, label='Estimated ensemble mean')

    ax1.plot(t, mu_x, linestyle='--', linewidth=1.8, label='Theoretical input mean')

    ax1.set_xlim(0, T_end)

    if mean_type == 'Constant':

        ax1.set_ylim(-3.5, 3.5)

    else:

        ax1.set_ylim(-4.0, 4.0)

    ax1.set_xlabel('Time t', fontsize=11)

    ax1.set_ylabel('x(t)', fontsize=11)

    ax1.set_title(f'Stochastic Input Process\n{mean_name}', fontsize=12, pad=9)

    ax1.tick_params(axis='both', labelsize=9)

    ax1.grid(True, linestyle=':', alpha=0.5)

    ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, fontsize=8, frameon=True)

    # ========================================================
    # GRAPH 2:
    # STOCHASTIC OUTPUT
    # ========================================================

    for m in range(number_to_show):

        ax2.plot(t, Y[m, :], linewidth=0.65, alpha=0.22)

    ax2.plot(t, estimated_mu_y, linewidth=2.0, label='Estimated output ensemble mean')

    ax2.plot(t, theoretical_mu_y, linestyle='--', linewidth=1.8, label='Response to input mean')

    ax2.set_xlim(0, T_end)

    if mean_type == 'Constant':

        ax2.set_ylim(-1.5, 2.5)

    else:

        ax2.set_ylim(-2.5, 2.5)

    ax2.set_xlabel('Time t', fontsize=11)

    ax2.set_ylabel('y(t)', fontsize=11)

    ax2.set_title('Output of the First-Order LTI System', fontsize=12, pad=9)

    ax2.tick_params(axis='both', labelsize=9)

    ax2.grid(True, linestyle=':', alpha=0.5)

    ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, fontsize=8, frameon=True)

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    plt.subplots_adjust(left=0.07, right=0.98, top=0.88, bottom=0.25, wspace=0.27)

    plt.show()
    plt.close(fig)

# ============================================================
# RADIO BUTTONS
# ============================================================

mean_selector = RadioButtons(options=['Constant', 'Sinusoidal'], value='Constant', description='', layout=Layout(width='230px'))

mean_selector.add_class('horizontal-radio')

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}

slider_layout = Layout(width='160px')

mean_slider = FloatSlider(min=0.0, max=2.0, step=0.1, value=1.0, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

tau_slider = FloatSlider(min=0.2, max=3.0, step=0.1, value=1.0, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

noise_slider = FloatSlider(min=0.0, max=2.0, step=0.1, value=0.8, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

M_slider = IntSlider(min=10, max=100, step=10, value=40, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

mean_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.0</div>')

tau_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.0</div>')

noise_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.8</div>')

M_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">40</div>')

empty_value = HTML('<div></div>')

# ============================================================
# UPDATE CURRENT VALUES
# ============================================================

def update_mean_value(change):

    mean_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{mean_slider.value:.1f}</div>'

def update_tau_value(change):

    tau_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{tau_slider.value:.1f}</div>'

def update_noise_value(change):

    noise_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{noise_slider.value:.1f}</div>'

def update_M_value(change):

    M_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{M_slider.value}</div>'

mean_slider.observe(update_mean_value, names='value')

tau_slider.observe(update_tau_value, names='value')

noise_slider.observe(update_noise_value, names='value')

M_slider.observe(update_M_value, names='value')

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(plot_stochastic_differential_equation, mean_type=mean_selector, mean_level=mean_slider, tau=tau_slider, noise_std=noise_slider, M=M_slider)

# ============================================================
# DOCUMENTATION
# ============================================================

theory_html = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:650px;
    padding:14px 16px;
    border:1px solid #c9b8dc;
    border-radius:8px;
    background:#fdfbff;
    box-sizing:border-box;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#65358d;
    margin-bottom:9px;
">
Mean Value and Stochastic Differential Equations
</div>

<div style="margin-bottom:6px;">
<b>System equation:</b>
τ dy(t)/dt + y(t) = x(t).
</div>

<div style="margin-bottom:6px;">
<b>Stochastic input:</b>
each realization x(t,ξᵢ) passes through the same deterministic LTI system.
</div>

<div style="margin-bottom:6px;">
<b>Mean equation:</b>
by applying mathematical expectation,
</div>

<div style="
    font-family:serif;
    font-size:21px;
    font-style:italic;
    color:#65358d;
    margin:8px 0px 8px 20px;
">
τ dμᵧ(t)/dt + μᵧ(t) = μₓ(t)
</div>

<div>
<b>This notebook:</b> compares the ensemble mean of the stochastic output with the deterministic system response obtained when μₓ(t) is used as the input.
</div>

</div>
""")

# ============================================================
# CONTROL LABELS
# ============================================================

mean_type_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Input mean:</div>')

mean_level_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Mean level A:</div>')

tau_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Time constant τ:</div>')

noise_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Noise std σ:</div>')

M_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Realizations M:</div>')

# ============================================================
# CONTROLS GRID
# ============================================================

controls_grid = GridBox(
    children=[
        mean_type_label, mean_selector, empty_value,
        mean_level_label, mean_slider, mean_value,
        tau_label, tau_slider, tau_value,
        noise_label, noise_slider, noise_value,
        M_label, M_slider, M_value
    ],
    layout=Layout(
        width='425px',
        grid_template_columns='135px 230px 45px',
        grid_template_rows='34px 34px 34px 34px 34px',
        grid_gap='4px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#65358d;
            margin-bottom:6px;
        ">
        Parameters
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='455px',
        min_width='455px',
        padding='14px 14px',
        border='1px solid #d2d2d2',
        overflow='hidden'
    )
)

# ============================================================
# TOP AREA
# ============================================================

top_layout = HBox(
    [
        theory_html,
        controls_card
    ],
    layout=Layout(
        width='1120px',
        align_items='stretch',
        justify_content='space-between'
    )
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation_html = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.42;
    width:1120px;
    padding:12px 16px;
    border:1px solid #d7cae2;
    background:#fdfbff;
    box-sizing:border-box;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#65358d;
    margin-bottom:7px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:5px;">
<b>Left:</b> the individual input realizations fluctuate randomly around the prescribed mean μₓ(t).
</div>

<div style="margin-bottom:5px;">
<b>Right:</b> the individual output realizations are different, but their ensemble mean follows the deterministic response produced by μₓ(t).
</div>

<div style="margin-bottom:5px;">
Increasing the number of realizations M reduces the random fluctuations of the estimated ensemble means.
</div>

<div style="
    margin-top:8px;
    font-family:serif;
    font-size:20px;
    font-style:italic;
    color:#65358d;
">
E{T[x(t)]} = T{E[x(t)]}
&nbsp;&nbsp;&nbsp;⇒&nbsp;&nbsp;&nbsp;
τ dμᵧ/dt + μᵧ = μₓ
</div>

</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        top_layout,
        widget_plot.children[-1],
        interpretation_html
    ],
    layout=Layout(
        width='1120px',
        overflow='hidden'
    )
)

display(main_layout)